# Model 1 - Issue Type Classifier (Pothole vs Garbage)
Notebook ini berisi alur sederhana dari awal sampai mendapatkan model `.pt` menggunakan PyTorch.

## 1. Setup dependency
Jalankan sekali di environment training Anda.

In [ ]:
# !pip install torch torchvision scikit-learn matplotlib


## 2. Import library

In [ ]:
import os
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


## 3. Struktur dataset
Gunakan `ImageFolder` dengan struktur:
```
dataset/
  train/
    class_a/
    class_b/
  val/
    class_a/
    class_b/
```

In [ ]:
DATA_DIR = Path('../../data/')  # ubah sesuai lokasi dataset Anda
TRAIN_DIR = DATA_DIR / 'train'
VAL_DIR = DATA_DIR / 'val'

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
] )

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_ds = datasets.ImageFolder(VAL_DIR, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

print('Classes:', train_ds.classes)
print('Train samples:', len(train_ds), '| Val samples:', len(val_ds))


## 4. Definisikan model
Gunakan ResNet18 pretrained agar cepat konvergen untuk MVP.

In [ ]:
num_classes = len(train_ds.classes)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


## 5. Training loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, running_correct = 0.0, 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == y).sum().item()
        total += y.size(0)
    return running_loss / total, running_correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, running_correct = 0.0, 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == y).sum().item()
        total += y.size(0)
    return running_loss / total, running_correct / total


In [ ]:
EPOCHS = 5
best_acc = 0.0
save_path = Path('../../ai/cv_model/weights') / 'issue_type_resnet18.pt'
save_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_acc = evaluate(model, val_loader, criterion)
    print(f'Epoch {epoch}: train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} val_loss={va_loss:.4f} val_acc={va_acc:.4f}')

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_names': train_ds.classes,
        }, save_path)

print('Best val acc:', best_acc)
print('Saved model to:', save_path)


## 6. Uji inferensi singkat

In [ ]:
from PIL import Image

ckpt = torch.load(save_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

sample_path = Path('../../sample.jpg')  # ganti dengan contoh gambar
img = Image.open(sample_path).convert('RGB')
x = val_tfms(img).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(x)
    probs = torch.softmax(logits, dim=1)[0]
    pred_idx = int(torch.argmax(probs).item())

print('Prediction:', ckpt['class_names'][pred_idx])
print('Confidence:', float(probs[pred_idx]))
